# 02 - Data Understanding## TujuanMemahami struktur, kelengkapan, dan kualitas data sebelum diproses lebih lanjut,serta mendokumentasikan dokumen yang dibuang beserta alasannya.## Asal (Input)`data/raw/thesis_scraped.csv` - 312 dokumen hasil Notebook 01## Keluaran (Output)- `data/processed/thesis_for_pipeline.csv` - 309 dokumen (filter awal)- `docs/data_lineage_by_year.csv` - jejak jumlah dokumen per tahap- `docs/excluded_documents.csv` - dokumen yang dibuang + alasannya## Pertanyaan yang Dijawab- Berapa dokumen yang punya field lengkap?- Dokumen mana yang terduplikasi judulnya?- Berapa dokumen yang lolos ke tahap berikutnya?

In [ ]:
import sys, io as _iofrom pathlib import Path_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()sys.path.insert(0, str(_ROOT))try:    sys.stdout.reconfigure(encoding='utf-8', errors='replace')except Exception:    passimport pandas as pd, numpy as npfrom src import config as CC.ensure_dirs()print(C.describe())

## Langkah 1 - Analisis Kelengkapan Field

In [ ]:
raw_df = pd.read_csv(C.SCRAPED_CSV)print(f'Dokumen : {len(raw_df)}')print(f'Kolom   : {list(raw_df.columns)}')print()for col in ['id', 'year', 'author', 'title', 'abstract']:    filled = raw_df[col].notna().sum()    print(f'  {col:10} terisi {filled:3d}/{len(raw_df)}  ({filled/len(raw_df)*100:5.1f}%)')

## Langkah 2 - Deteksi Duplikasi Judul

In [ ]:
raw_df['tmp_title_lower'] = raw_df['title'].astype(str).str.strip().str.lower()dupes = raw_df[raw_df.duplicated(subset=['tmp_title_lower'], keep='first')]print(f'Dokumen dengan judul duplikat: {len(dupes)}')if len(dupes):    for _, r in dupes.head(5).iterrows():        print(f'  ID {r["id"]:>7} ({r["year"]}): {str(r["title"])[:65]}')

## Langkah 3 - Filter Awal dan Lineage

In [ ]:
from src import data_lineage as dldl.run_data_lineage(verbose=True)

## Ringkasan Fase 02 - Data Understanding| Tahap | Jumlah Dokumen ||:---|---:|| Hasil web scraping | 312 || Setelah filter awal | 309 || **Korpus bersih akhir** | **285** |**Temuan utama:**1. Sebagian dokumen tidak memiliki teks abstrak sehingga tidak dapat dianalisis.2. Duplikasi judul diselesaikan dengan case-insensitive matching.3. Setelah preprocessing, 285 dokumen lolos dengan minimal 5 token per dokumen.

## Peta Artefak| Artefak | Dibuat oleh | Consumed by ||:---|:---|:---|| `data/processed/thesis_for_pipeline.csv` | Notebook 02 | Notebook 03 || `docs/data_lineage_by_year.csv` | Notebook 02 | Lampiran Bab III || `docs/excluded_documents.csv` | Notebook 02 | Lampiran Bab III |